# Getting Started — OptimalCover VSC Pricing Dataset

This notebook loads every table in the dataset and shows you what's inside. Run top-to-bottom.

**License:** CC BY 4.0 — attribution required. See `LICENSE` at the repo root.

**Schema version:** v3.6.7

In [ ]:
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/Optimal-Cover/vsc-pricing-data/main/data/v3.6.7"

rates        = pd.read_csv(f"{BASE_URL}/vsc_rates.csv")
eligibility  = pd.read_csv(f"{BASE_URL}/vehicle_eligibility.csv")
mileage_bands = pd.read_csv(f"{BASE_URL}/mileage_bands.csv")
config       = pd.read_csv(f"{BASE_URL}/pricing_config.csv")

print(f"rates:        {rates.shape[0]} rows x {rates.shape[1]} cols")
print(f"eligibility:  {eligibility.shape[0]} rows x {eligibility.shape[1]} cols")
print(f"mileage_bands: {mileage_bands.shape[0]} rows x {mileage_bands.shape[1]} cols")
print(f"config:       {config.shape[0]} rows x {config.shape[1]} cols")

## 1. Reference prices (`vsc_rates`)

One row per `(vehicle_class, mileage_band, term, band)`. `band` is either `REFERENCE_LOW` or `REFERENCE_HIGH` — pair them to form a price range.

In [ ]:
rates.head(8)

## 2. Vehicle eligibility (`vehicle_eligibility`)

Each make is mapped to a risk class (A–D). `models` is a JSON-encoded list in CSV/Parquet, a native array in JSON.

In [ ]:
eligibility[['vehicle', 'vehicle_class', 'eligibility']].head(10)

## 3. Mileage bands (`mileage_bands`)

Definitions of the bands referenced by `vsc_rates.mileage_band`.

In [ ]:
mileage_bands

## 4. Methodology config (`pricing_config`)

Encoded constants — schema version, deductible rules, constraints, rounding policy.

In [ ]:
config[['config_key']]

## 5. Pair REFERENCE_LOW and REFERENCE_HIGH to form a price range

A single scenario is the pair of rows sharing `(vehicle_class, mileage_band, term_months, deductible)`.

In [ ]:
ranges = rates.pivot_table(
    index=['vehicle_class', 'mileage_band', 'term_months', 'deductible'],
    columns='band',
    values='price',
    aggfunc='first',
).reset_index()

ranges['spread'] = ranges['REFERENCE_HIGH'] - ranges['REFERENCE_LOW']
ranges.head(10)

---

**Next:** see `02_pricing_explorer.ipynb` for analysis and visualizations, and `03_deductible_comparison.ipynb` for how `$0` and `$250` deductibles are derived from the canonical `$100`.